# Input format
- ## LPI pairs  
RNA	protein  
AK006025	B2RWS6  
AK020562	P45481  
AK020562	Q78PY7  

- ## kmer matrix  
		AAA	AAC	AAG	AAU ...
AK028540_Q8R081	0.040566398775354	0.0191350937619594	0.0229621125143513	0.0176042862610026 ...  
AK020562_Q78PY7	0.0346215780998389	0.0120772946859903	0.0209339774557165	0.0289855072463768 ...  
NR_110420_O88569	0.0476190476190476	0.0195065978198508	0.0344234079173838	0.0246701090074584 ...


In [1]:
import warnings
warnings.filterwarnings("ignore")


import pandas as pd
import numpy as np
import math
import os
import sys
import time
import configparser
from argparse import ArgumentParser
import itertools
import math

import textwrap

from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

import torch
from torch import nn
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score, recall_score, precision_score, accuracy_score, f1_score
from sklearn.model_selection import GroupKFold, KFold

from lpitabformer.utils import *
from lpitabformer.ft_transformer import *
from lpitabformer import ft_transformer


In [2]:
parser = ArgumentParser()
#parser.add_argument('--datadir', type=str, default='C:/Users/Ci/JupyterLabWork/LPITransformer/DsuLinformer/compare/data/project230830/feature', help='The dataset directory you want to process.')
parser.add_argument('--dataset', type=str, default='tool_sample.txt', help='The dataset you want to process.') # LPI500684 LPI300146_seed1_randp1
parser.add_argument('--workdir', type=str, default="./", help='The initial directory.')
parser.add_argument('--bs', type=int, default=5, help='Batch size')

parser.add_argument('--seed', type=int, default=1, help='random seed')
parser.add_argument('--mode', type=int, default=0, help='Mode for category and numeric for RNA and protein kmer. 0 is rna-cat and prot-num; 1 is rna-num and prot-cat; 2 is rna-cat and prot-cat; 3 is rna-num and pro-num.') 

parser.add_argument('--rKmer', type=str, default="./data/3mer_tool_sample.csv", help="All lncRNA kmer path") #20240127
parser.add_argument('--pKmer', type=str, default="./data/1mer_tool_sample.csv", help="All protein kmer path") #20240127
parser.add_argument('--clusterPath', type=str, default="./data/others/UP000005640_79740_uniprot_hsa_iden0.5_cluster.tsv", help='protein cluster file.')

##20240503
parser.add_argument('--modelPath', type=str, default="", help='trained model directory')
parser.add_argument('--lpi', default=None, help="Name of LPI for predicting") #20240127
parser.add_argument('--lpi_file', default="./data/tool_sample.csv", help="Preprocessed matrix for lpi kmer") #20240127

parser.add_argument('--use_kmerM', action='store_false', default=True, help="Whether use the kmer matrix for input. If False, you need input the LPI pairs and waiting for kmer matrix generation")



_StoreFalseAction(option_strings=['--use_kmerM'], dest='use_kmerM', nargs=0, const=False, default=True, type=None, choices=None, required=False, help='Whether use the kmer matrix for input. If False, you need input the LPI pairs and waiting for kmer matrix generation', metavar=None)

In [3]:
## Using kmer matrix
def get_param():
    global args
    args = parser.parse_args(['--mode', '0',
                              '--dataset', 'tool_sample',
                              '--lpi', "tool_sample.csv",
                              '--modelPath', "./data/model/gkfold_mus.pth",
                             ]) # 15 epoches for test
    return args


In [4]:
import warnings

warnings.filterwarnings("ignore")
import sys

import pandas as pd
import numpy as np
import os

from datetime import datetime
from torch.utils.data import DataLoader, TensorDataset

import torch

from joblib import Parallel, delayed
import multiprocessing


def retrVal(indx, df, rna_mer, pro_mer, sep="_"):
    global rkmer, pkmer
    # rna = df.loc[indx][0]
    rna = df.loc[indx].iloc[0]
    pro = df.loc[indx].iloc[1]
    rkmer = rna_mer.loc[rna].values
    pkmer = pro_mer.loc[pro].values
    rp = np.concatenate((rkmer, pkmer), axis=0)
    rp_tuple = (rna + str(sep) + pro, rp)
    return rp_tuple


def lpi2data(df, rna_mer, pro_mer, sep="_", num_cores=multiprocessing.cpu_count(), backend="threading"):
    global kmer_list, kdf
    # kdf = pd.DataFrame(columns=np.concatenate((rna_mer.columns, pro_mer.columns), axis=0))
    '''
    backend:
    threading: multi threads for I/O task.
    multiprocessing and loky: for CPU enriched task.
    '''
    kmer_list = Parallel(n_jobs=num_cores, backend=backend) \
        (delayed(retrVal) \
             (indx=indx, df=df, rna_mer=rna_mer, pro_mer=pro_mer, sep=sep) \
         for indx in range(len(df)))

    kdf = pd.DataFrame(dict(kmer_list)).T
    kdf.columns = np.concatenate((rna_mer.columns, pro_mer.columns), axis=0)

    print(kdf.head())

    return kdf


def preprocess(args):
    global lpi, data, group_df, groups, gdf, rna_mer, pro_mer

    if "gencode" in args.clusterPath or "NONCODE" in args.clusterPath:
        select_part = 'front'
        merge_name = "RNA"
    else:
        select_part = 'back'
        merge_name = "protein"

    def transform(x, sep="_", select_part=select_part):
        if select_part == "back":
            x = x.split(sep)[1]  # Note: RNA.protein is formated in LION feature matrix
        else:
            x = x.split(sep)[0]
        return x

    rna_mer = pd.read_csv(args.rKmer, header=0, index_col=0,
                          usecols=lambda column: column != 'label')  # col 'label' is in MathFeature
    pro_mer = pd.read_csv(args.pKmer, header=0, index_col=0, usecols=lambda column: column != 'label')

    print(rna_mer.shape)

    lpi = pd.read_csv(args.lpi_file, header=0, index_col=0)
    if args.use_kmerM:
        data = lpi
    else:
        data = lpi2data(lpi, rna_mer, pro_mer, num_cores=12)  # -1 to 24


def dataPre(args):
    global tuple_cat, num_conti, col_names, data
    # NPInter2 and other datasets is RNA-protein-label, but RPI488 and RPI1807 is protein-RNA-label.
    if args.mode == 0:
        col_cat = ['C' + str(i) for i in range(1, 65)]
        col_num = ['N' + str(i) for i in range(1, 21)]
        col_names = col_cat + col_num
        tuple_cat = tuple([64] * 64)
        num_conti = 20
    elif args.mode == 1:
        col_cat = ['C' + str(i) for i in range(1, 21)]
        col_num = ['N' + str(i) for i in range(1, 65)]
        col_names = col_num + col_cat
        tuple_cat = tuple([20] * 20)
        num_conti = 64
    elif args.mode == 2:
        col_cat = ['C' + str(i) for i in range(1, 85)]
        col_num = ['N' + str(i) for i in range(1, 2)]
        col_names = col_cat + col_num
        tuple_cat = tuple([64] * 64 + [20] * 20)
        num_conti = 1
    elif args.mode == 3:
        col_cat = ['C' + str(i) for i in range(1, 2)]
        col_num = ['N' + str(i) for i in range(1, 85)]
        col_names = col_cat + col_num
        tuple_cat = tuple([1] * 1)
        num_conti = 84

    if args.mode == 2:
        data.loc[:, "N1"] = 0
        data.columns = col_names
    elif args.mode == 3:
        data.columns = col_num
        data.loc[:, "C1"] = 0
        data = data[col_names]
    else:
        data.columns = col_names

    print(data.head())

    data = data[col_cat + col_num]

    print(data.head())

    data[col_cat] = data[col_cat].fillna('0', )
    data[col_num] = data[col_num].fillna('0', )

    if args.mode == 2:
        cat_rna = ['C' + str(i) for i in range(1, 65)]
        cat_prot = ['C' + str(i) for i in range(65, 85)]
        ###RNA
        df_rna = data[cat_rna]
        for ind in range(df_rna.shape[0]):
            # df_cat.iloc[ind] = df_cat.iloc[ind].rank().astype(np.int32) - 1 #Note minus 1
            df_rna.iloc[ind] = np.argsort(df_rna.iloc[ind]).astype(np.int32)

        data[cat_rna] = df_rna.astype(np.int32)
        ###protein
        df_prot = data[cat_prot]
        for ind in range(df_prot.shape[0]):
            # df_cat.iloc[ind] = df_cat.iloc[ind].rank().astype(np.int32) - 1 #Note minus 1
            df_prot.iloc[ind] = np.argsort(df_prot.iloc[ind]).astype(np.int32)

        data[cat_prot] = df_prot.astype(np.int32)
    else:
        df_cat = data[col_cat]
        for ind in range(df_cat.shape[0]):
            # df_cat.iloc[ind] = df_cat.iloc[ind].rank().astype(np.int32) - 1 #Note minus 1
            df_cat.iloc[ind] = np.argsort(df_cat.iloc[ind]).astype(np.int32)

        data[col_cat] = df_cat.astype(np.int32)


def get_model():
    global ftt, loss_func
    ftt = FTTransformer(
        categories=(10, 5, 6, 5, 8),  # tuple containing the number of unique values within each category
        num_continuous=10,  # number of continuous values
        dim=32,  # dimension, paper set at 32
        dim_out=1,  # binary prediction, but could be anything
        depth=6,  # depth, paper recommended 6
        heads=8,  # heads, paper recommends 8
        attn_dropout=0.1,  # post-attention dropout
        ff_dropout=0.1,  # feed forward dropout
        p=0,

    )

    x_categ = torch.randint(0, 5, (
        1, 5))  # category values, from 0 - max number of categories, in the order as passed into the constructor above
    x_numer = torch.randn(1, 10)  # numerical value
    print(x_categ.shape)
    print(x_numer.shape)
    ftt(x_categ, x_numer)  # (1, 1)


def get_predict(loader, model, device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'), mode=0):
    pred, target = [], []
    model.eval()
    with torch.no_grad():
        for x in loader:
            x = x[0].to(device).float()
            if mode == 0:
                x_categ = x[:, :64].cuda().to(torch.long)
                x_numer = x[:, 64:].cuda().to(torch.float32)
            elif mode == 1:
                x_categ = x[:, :20].cuda().to(torch.long)
                x_numer = x[:, 20:].cuda().to(torch.float32)
                # elif mode == 2:
            # 20240127
            elif mode == 2:
                x_categ = x[:, :84].cuda().to(torch.long)
                x_numer = x[:, 84:].cuda().to(torch.float32)
            elif mode == 3:
                x_categ = x[:, :1].cuda().to(torch.long)  # add in 20231121
                x_numer = x[:, 1:].cuda().to(torch.float32)

            y_hat = model(x_categ, x_numer)

            pred += list(y_hat.cpu().numpy())

    pred = pd.DataFrame(
        torch.tensor(np.array(pred)).softmax(dim=-1).numpy())  # torchmetrics will auto-determine logits or probs.
    lpi_index = pd.DataFrame(data.index, columns=['LPI'])
    probs_lpi = pd.concat([lpi_index, pred], axis=1)

    return probs_lpi


def modelPredict(args, data, mode, device, time=1234,
                 modelPath=None):
    global test_loader, evalCol, outDir, resDir, outDir
    resDir = os.path.join(args.workdir, "data", "result", args.dataset)
    if os.path.exists(resDir):
        print(f"{resDir} exits!")
    else:
        os.makedirs(resDir)

    outDir = os.path.join(resDir, "test")
    if os.path.exists(outDir):
        print(f"{outDir} exits!")
    else:
        os.makedirs(outDir)

    test_tensor_data = TensorDataset(torch.from_numpy(np.array(data)))
    test_loader = DataLoader(test_tensor_data, batch_size=args.bs)

    print("======================", "Start!", "======================", "\n")
    model = torch.load(modelPath)

    probs_lpi = get_predict(test_loader, model, device, mode)  # get_result3 is for FTTransformer

    print("Model:", modelPath, "Test dataset:", data_name, "\n")
    probs_lpi.to_csv(
        os.path.join(outDir, "te_res_" + data_name  + "_" + str(time) + ".csv"),
        index=True, header=True)


class Args:
    def __init__(self, **kwargs):
        for key, value in kwargs.items():
            setattr(self, key, value)

In [5]:
%%time
if __name__ == '__main__':
    get_param()
    fix_random_seed(args.seed)
    
    global data_name, device
    
    #######################
    os.chdir(args.workdir)
    #######################
    
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++","\n")
    print("++++++++++++++++++", "seed", args.seed, "start!", "++++++++++++++++++","\n")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++++","\n")
        
    data_name = args.dataset
    ##0 prepare
    os.environ['CUDA_VISIBLE_DEVICES'] = '0' 
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    #device ="cpu"
    print("Device available: ", device, " ", torch.cuda.get_device_name(0))
        
    ##1
    preprocess(args)
    dataPre(args)
    get_model()
    data
        
    from datetime import datetime
    global now_time
    now_time = datetime.now().strftime('%Y_%m_%d_%H%M%S_%f')
        
    modelPredict(args, data=data, mode=args.mode, device=device, time=now_time, modelPath=args.modelPath)
    
    print(args.dataset, args.seed, "\n", "Seed", args.seed,  args.mode, "OK!", "\n")
    print("---------------------------------------------------------------------","\n")
    print("---------------------------------------------------------------------","\n")
        

+++++++++++++++++++++++++++++++++++++++++++++++++++ 

++++++++++++++++++ seed 1 start! ++++++++++++++++++ 

+++++++++++++++++++++++++++++++++++++++++++++++++++ 

Device available:  cuda   NVIDIA GeForce RTX 3080
(3, 64)
                        C1        C2        C3        C4        C5        C6  \
AK028540_Q8R081   0.040566  0.019135  0.022962  0.017604  0.033678  0.013012   
AK020562_Q78PY7   0.034622  0.012077  0.020934  0.028986  0.023349  0.010467   
NR_110420_O88569  0.047619  0.019507  0.034423  0.024670  0.020080  0.004016   

                        C7        C8        C9       C10  ...       N11  \
AK028540_Q8R081   0.003062  0.021049  0.023728  0.019900  ...  0.025597   
AK020562_Q78PY7   0.004831  0.017713  0.026570  0.012882  ...  0.015385   
NR_110420_O88569  0.001147  0.019507  0.035571  0.016064  ...  0.019830   

                       N12       N13       N14       N15       N16       N17  \
AK028540_Q8R081   0.058020  0.083618  0.044369  0.059727  0.069966  0.030717  